# 34 — Critical Slowing Down Near FIM Transition

**UNPRECEDENTED:** Near a phase transition, the system's relaxation time
diverges — perturbations take longer to decay. This is "critical slowing
down" (CSD). It's used as an early warning signal for tipping points
in climate, ecology, and neuroscience.

NB43 showed SCPN has BKT universality (β→0). For BKT transitions,
the correlation time diverges as τ ~ exp(c/√(K-K_c)) — faster than
any power law.

**Questions:**
1. Does CSD exist in FIM-Kuramoto?
2. Does FIM change the CSD signature? (stronger/weaker/different)
3. Can CSD be used as an early warning for consciousness transitions?

## Method
Autocorrelation time τ_R of the order parameter R(t):
C(Δt) = <R(t)R(t+Δt)> - <R>²
τ_R = integral of C(Δt)/C(0)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import OMEGA_N_16, build_knm_paper27

N = 16
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]


def fim_gradient_all(phases):
    n = len(phases)
    z = np.exp(1j * phases)
    mu = np.angle(np.mean(z))
    R = np.abs(np.mean(z))
    phase_diff = (mu - phases + np.pi) % (2 * np.pi) - np.pi
    sensitivity = min(1.0 / (1.0 - R**2 + 0.01), 50.0)
    return (1.0 / n) * np.sin(phase_diff) * sensitivity


def simulate_R_timeseries(K_scale, lam, dt=0.02, T=500.0, noise=0.05, seed=42):
    """Return full R(t) timeseries for autocorrelation analysis."""
    K = K_base * K_scale
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, N)
    n_steps = int(T / dt)
    sample_every = 5

    R_series = []
    for s in range(n_steps):
        diff = theta[None, :] - theta[:, None]
        coupling = np.sum(K * np.sin(diff), axis=1) / N
        dphi = omega + coupling
        if lam > 0:
            dphi += lam * fim_gradient_all(theta)
        theta = (theta + dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)) % (2 * np.pi)
        if s % sample_every == 0 and s >= n_steps // 4:
            R_series.append(float(np.abs(np.mean(np.exp(1j * theta)))))

    return np.array(R_series), dt * sample_every


def autocorrelation_time(R_series):
    """Compute integrated autocorrelation time."""
    R = R_series - np.mean(R_series)
    n = len(R)
    C0 = np.var(R)
    if C0 < 1e-10:
        return 0.0  # no fluctuations = no slowing down

    # Autocorrelation via FFT
    F = np.fft.fft(R, n=2 * n)
    acf = np.real(np.fft.ifft(F * np.conj(F)))[:n] / (n * C0)

    # Integrated autocorrelation: sum until first negative
    tau = 0.5  # start at 0.5 (convention)
    for i in range(1, n // 2):
        if acf[i] < 0.05:  # cutoff at 5% correlation
            break
        tau += acf[i]

    return float(tau)


print("Ready.")

In [ ]:
# Sweep K near K_c for different λ
K_sweep = np.linspace(3, 20, 18)

print("=== CRITICAL SLOWING DOWN ===")
for lam in [0.0, 1.0, 3.0]:
    print(f"\n--- λ = {lam} ---")
    print(f"{'K':>6} {'<R>':>8} {'Var(R)':>10} {'τ_R':>8} {'CSD?':>6}")

    tau_values = []
    R_values = []
    for K_scale in K_sweep:
        R_ts, dt_sample = simulate_R_timeseries(K_scale, lam)
        R_mean = np.mean(R_ts)
        R_var = np.var(R_ts)
        tau = autocorrelation_time(R_ts)
        tau_values.append(tau)
        R_values.append(R_mean)
        csd = "YES" if tau > 5 else "no"
        print(f"{K_scale:6.1f} {R_mean:8.4f} {R_var:10.6f} {tau:8.2f} {csd:>6}")

    # Find peak tau (= K_c location)
    peak_idx = np.argmax(tau_values)
    print(f"  Peak τ = {tau_values[peak_idx]:.2f} at K = {K_sweep[peak_idx]:.1f}")
    print(f"  R at peak: {R_values[peak_idx]:.4f}")

In [ ]:
# Does FIM change CSD magnitude?
print("\n=== FIM EFFECT ON CSD ===")
# At K closest to K_c for each lambda
K_near_Kc = {0.0: 10.0, 1.0: 8.0, 3.0: 5.0}  # approximate K_c values

for lam, K_c_approx in K_near_Kc.items():
    tau_vals = []
    for K_offset in [-2, -1, 0, 1, 2]:
        K_s = max(1, K_c_approx + K_offset)
        R_ts, _ = simulate_R_timeseries(K_s, lam, T=500)
        tau = autocorrelation_time(R_ts)
        tau_vals.append((K_s, tau))

    peak = max(tau_vals, key=lambda x: x[1])
    print(f"λ={lam}: peak τ = {peak[1]:.2f} at K={peak[0]:.1f}")

# Variance as early warning
print("\n=== VARIANCE AS EARLY WARNING ===")
print("Variance of R should peak near K_c (fluctuation-dissipation)")
for lam in [0.0, 1.0, 3.0]:
    var_max = 0
    K_var_peak = 0
    for K_scale in K_sweep:
        R_ts, _ = simulate_R_timeseries(K_scale, lam, T=300)
        v = np.var(R_ts)
        if v > var_max:
            var_max = v
            K_var_peak = K_scale
    print(f"λ={lam}: Var(R) peaks at K={K_var_peak:.1f} (Var={var_max:.6f})")

print("\n=== CLINICAL RELEVANCE ===")
print("CSD could serve as early warning for consciousness transitions:")
print('- Rising τ_R → approaching sync threshold → "waking up"')
print("- Rising Var(R) → approaching transition → unstable state")
print("- Measurable from EEG order parameter time series")

In [ ]:
# Save
output = {"experiment": "Critical slowing down near FIM transition", "N": N}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/critical_slowing_down_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Results saved to {out_path}")